# PMVC-WNM: Phonetic Multi-View Co-training
### Robust Banglish Text Classification with Weakly Supervised Noise Modeling
**CSE 4622 — Machine Learning Lab | IUT | Team Open Source**

---

## Step 1: Setup & Install

In [ ]:
# Install dependencies
!pip install scikit-learn pandas numpy matplotlib seaborn -q

# Clone or mount project
# Option A: if using Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/pmvc_wnm

# Option B: clone from GitHub (if uploaded)
# !git clone https://github.com/YOUR_USERNAME/pmvc_wnm.git
# %cd pmvc_wnm

# Option C: upload project zip
# from google.colab import files
# uploaded = files.upload()  # upload pmvc_wnm.zip
# !unzip pmvc_wnm.zip
# %cd pmvc_wnm

print('Setup complete')

## Step 2: Upload Dataset

In [ ]:
from google.colab import files
import pandas as pd
import os

# Upload the Kaggle CSV
# Download from: https://www.kaggle.com/datasets/deepz99/b-and-b-80k
uploaded = files.upload()

# Get filename
csv_name = list(uploaded.keys())[0]
print(f'Uploaded: {csv_name}')

# Move to data folder
os.makedirs('data', exist_ok=True)
os.rename(csv_name, f'data/{csv_name}')
CSV_PATH = f'data/{csv_name}'

## Step 3: Preprocess

In [ ]:
import sys
sys.path.insert(0, '.')

from src.preprocess import load_and_clean

df = load_and_clean(CSV_PATH)
print(df.head())
print(f'\nShape: {df.shape}')
print(f'\nLabel distribution:')
print(df['label'].value_counts())

## Step 4: Feature Extraction — View A (char n-grams)

In [ ]:
from src.view_a_ngram import build_view_a

X_A, vec_A = build_view_a(df['clean'].tolist())
print(f'View A shape: {X_A.shape}')
print(f'Example features: {vec_A.get_feature_names_out()[:10]}')

## Step 5: Feature Extraction — View B (phonetic codes)

In [ ]:
from src.view_b_phonetic import build_view_b, encode_text

X_B, vec_B, encoded = build_view_b(df['clean'].tolist())
print(f'View B shape: {X_B.shape}')

# show example encoding
example = df['clean'].iloc[0]
phon, word = encode_text(example)
print(f'\nExample text:        {example}')
print(f'Phonetic (sub-word):  {phon}')
print(f'Phonetic (word-id):   {word}')

## Step 6: Run PMVC-WNM Full Training

In [ ]:
from src.cotraining import PMVCTrainer
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

trainer = PMVCTrainer(
    seed_size=500,
    T=10,
    K=100,
    threshold=0.75,
    t_start=3,
    noise_rate=0.20
)

trainer.fit(train_df.reset_index(drop=True))

## Step 7: Evaluate — Classification Report

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

test_texts = test_df['clean'].tolist()
y_test = test_df['label'].values

preds = trainer.predict(test_texts)

print('=== PMVC-WNM Results ===')
print(f'Macro F1:  {f1_score(y_test, preds, average="macro"):.4f}')
print(f'Accuracy:  {accuracy_score(y_test, preds):.4f}')
print()
print(classification_report(y_test, preds))

## Step 8: Ablation Study

In [ ]:
from src.evaluate import run_ablation, plot_ablation

print('Running ablation (this will take a few minutes)...')
results_df, trained_model = run_ablation(df, seed_size=500)

print(results_df[['model','f1','accuracy']].to_string(index=False))
plot_ablation(results_df)

## Step 9: Learning Curve

In [ ]:
from src.evaluate import run_learning_curve, plot_learning_curve

print('Running learning curve (this will take several minutes)...')
seed_sizes, svm_scores, pmvc_scores = run_learning_curve(
    df, seed_sizes=[100, 250, 500, 1000, 2000]
)

plot_learning_curve(seed_sizes, svm_scores, pmvc_scores)

## Step 10: Noise Robustness Test

In [ ]:
from src.evaluate import run_noise_robustness

clean_f1, noisy_f1 = run_noise_robustness(trainer, df)

## Step 11: Disagreement Rate Convergence

In [ ]:
from src.evaluate import plot_disagree_rate

plot_disagree_rate(trainer.disagree_rates)

## Step 12: Spelling Error Matrix (Top Distortions)

In [ ]:
distortions = trainer.noise_model.get_top_distortions(top_n=5)

print('=== Spelling Error Matrix ===')
print('Phonetic Code  →  Observed Spellings (probability)')
print('─' * 55)
for ph_code, variants in list(distortions.items())[:10]:
    variants_str = ', '.join([f'{sp}({p:.2f})' for sp, p in variants])
    print(f'{ph_code:15s} →  {variants_str}')

---
## Done
All results generated. Use plots for your defense presentation.